In [1]:
import os
import re
from typing import List, Optional

import pandas as pd
from transformers import AutoTokenizer, AutoModel
import sentencepiece as spm


d:\conda_envs\deep-past\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# paths of files provided by competition host
PUBLISHED_TEXTS_CSV_PATH = os.path.join("data", "published_texts.csv")
TRAIN_CSV_PATH = os.path.join("data", "train.csv")

# paths of output files
TOKENIZER_TRAINING_CORPUS_PATH = os.path.join("data", "clean_akkadian_corpus.txt")

# model folder
MODEL_FOLDER = "models"
MODEL_PREFIX = "mt5-base-finetuned-english-akkadian"

# Data Cleaning

In [ ]:
class AkkadianTextCleaner:
    """
    Class for cleaning akkadian text
    It mainly does these:
    1. Remove invalid characters
    2. Remove redundant spaces
    3. Normalize special characters (subscripts, characters)
    """
    def __init__(self):

        self.diacritic_mapping = {
            'sz': 'š', # convert ASCII substitutes (e.g., sz) into diacritics
            'SZ': 'Š',
            'ṣ': 'ṣ',  # keep diacritics
            'ṭ': 'ṭ',
            'Š': 'Š',
            'š': 'š',
            'Ṣ': 'Ṣ',
            'ṭ': 'ṭ',
            'Ṭ': 'Ṭ'
        }
        
        # Gap normalization
        ## basic
        self.gap_mapping = {
            r'\[x\]': '<gap>',          # [x] → <gap>
            r'…': '<big_gap>',          # …  → <big_gap>
            r'\[\…\s*\…\]': '<big_gap>' # [… …] → <big_gap>（
        }
        self.gap_regex = re.compile('|'.join(self.gap_mapping.keys()))

        ## concatenated <gap> or <big_gap>
        self.gap_hyphen_pattern = re.compile(r'(-?<(gap|big_gap)>-?)')
        self.combined_gap_patterns = [
            r'<gap>\s*<big_gap>', r'<big_gap>\s*<gap>', r'<gap>\s*<gap>', r'<big_gap>\s*<big_gap>'
        ]
        self.combined_gap_regex = re.compile('|'.join(self.combined_gap_patterns))
        
        self.akkadian_residue_pattern = re.compile(r'<(big_)?gap>|\{[a-zA-Z0-9₂₃]*\}')
        
        # remove invalid characters
        self.invalid_patterns = [
            r'[!?:./\\˹˺]',  # 孤立现代标点/标记
            r'\([^)]*\)',     # 圆括号内注释（如(scholar note)、(11:11)）
            r'\[[^x\…]*\]'    # 方括号内非缺口标记（如[破损]、[注释]，排除[x]、[… …]）
        ]
        self.invalid_regex = re.compile('|'.join(self.invalid_patterns))
        
        # subscripts
        self.subscript_num_pattern = re.compile(r'([a-zA-ZšŠṣṢṭṬáàéèíìúù])[₀₁₂₃₄₅₆₇₈₉](?![a-zA-Z])')
        self.plain_num_subscript_pattern = re.compile(r'([a-zA-ZšŠṣṢṭṬáàéèíìúù])([23])(?![a-zA-Z])')

    def clean_single_text(self, text: str) -> Optional[str]:
        """
        Main method
        """
        if not isinstance(text, str) or text.strip() == "":
            return None
        
        text = self._remove_invalid_strings(text)
        text = self._normalize_special_formats(text)
        text = self._merge_combined_gaps(text)
        text = self._clean_extra_spaces(text)
        
        # 最终过滤：空文本或仅含空白字符的文本
        if not text or text.strip() == "":
            return None
        
        return text

    def _remove_invalid_strings(self, text: str) -> str:
        if not isinstance(text, str):
            return ""
        text = self.invalid_regex.sub('', text)
        return text

    def _normalize_special_formats(self, text: str) -> str:
        """
        Do the followings:
        1. replace gap markings to <gap>/<big_gap>: [x] -> <gap>, …/[… …] -> <big_gap>
        2. subscripts: ₂ or ₃ -> diacritics, others-> normal numbers
        3. ASCII substitutes -> diacritics, e.g. sz -> š
        """
        if not isinstance(text, str):
            return ""
        
        # replace gap markings to <gap>/<big_gap>
        def _replace_gap(match):
            return self.gap_mapping.get(match.group(0), match.group(0))
        text = self.gap_regex.sub(_replace_gap, text)
        
        # handle subscripts
        # turn everything into normal number first (including 2 and 3)
        def _replace_subscript(match):
            char = match.group(1)
            subscript = match.group(0)[-1]
            subscript_to_plain = {'₀':'0','₁':'1','₂':'2','₃':'3','₄':'4','₅':'5','₆':'6','₇':'7','₈':'8','₉':'9'}
            plain_num = subscript_to_plain.get(subscript, subscript)
            return char + plain_num
        text = self.subscript_num_pattern.sub(_replace_subscript, text)
        
        # then handle 2 -> ú, 3->ù
        # target pattern: character+2/3 
        text = re.sub(r'([a-zA-ZšŠṣṢṭṬ])2(?![0-9])', r'\1ú', text)
        text = re.sub(r'([a-zA-ZšŠṣṢṭṬ])3(?![0-9])', r'\1ù', text)
        
        # 3
        for old_str, new_str in self.diacritic_mapping.items():
            text = text.replace(old_str, new_str)
        
        return text

    def _merge_combined_gaps(self, text: str) -> str:
        if not isinstance(text, str):
            return ""
        
        # temporarily replace the hyphens with marker, e.g. "-<gap>" -> __HYPHEN__<gap>，"<gap>-" -> <gap>__HYPHEN__
        def _strip_hyphen(match):
            full_match = match.group(0)
            gap_tag = match.group(2)

            has_prefix_hyphen = full_match.startswith('-')
            has_suffix_hyphen = full_match.endswith('-')

            temp_gap = f"{ '__HYPHEN__' if has_prefix_hyphen else '' }<{gap_tag}>{ '__HYPHEN__' if has_suffix_hyphen else '' }"
            return temp_gap
        # 替换所有“连字符+缺口”为“临时标记+缺口”
        text = self.gap_hyphen_pattern.sub(_strip_hyphen, text)

        # 步骤2：合并纯缺口组合（忽略空格干扰）
        def _replace_combined_gap(match):
            matched_str = match.group(0)
            return '<big_gap>' if 'big_gap' in matched_str else '<gap>'
        # 先去除缺口间的空格，再合并
        text = re.sub(r'<(gap|big_gap)>\s*<(gap|big_gap)>', r'<\1><\2>', text)
        text = self.combined_gap_regex.sub(_replace_combined_gap, text)

        # 步骤3：还原连字符附着（将临时标记替换回连字符）
        text = text.replace('__HYPHEN__', '-')

        return text

    def _clean_extra_spaces(self, text: str) -> str:
        
        if not isinstance(text, str):
            return ""
        
        # { d }→{d}、{ lu }→{lu}
        text = re.sub(r'\s*{\s*', '{', text)
        text = re.sub(r'\s*}\s*', '}', text)
        
        # <gap> -A-šùr→<gap>-A-šùr
        text = re.sub(r'<(gap|big_gap)>\s+', r'<\1>', text)
        text = re.sub(r'\s+<(gap|big_gap)>', r'<\1>', text)
        
        # "    " -> " "
        text = re.sub(r'\s+', ' ', text).strip()
        
        return text

    

# Training the Tokenizer

### Prepare data

I plan to train the tokenizer on the transliteration data in `published_texts.csv`. 

The data has already been cleaned.

In [3]:
published_texts_df = pd.read_csv(PUBLISHED_TEXTS_CSV_PATH)

In [4]:
published_texts_df.head()

,oare_id,online transcript,cdli_id,aliases,label,publication_catalog,description,genre_label,inventory_position,online_catalog,note,interlinear_commentary,online_information,excavation_no,oatp_key,eBL_id,AICC_translation,transliteration_orig,transliteration
0,bd4b7138-70d5-490d-8234-299b0a75d3a3,https://oare.byu.edu/epigraphies/bd4b7138-70d5...,P361099,MP 1 bis,Cuneiform Tablet RA 60 125,RA 60 125,"Cuneiform envelope, dated to the Old Assyrian ...",letter,MP 1 bis,NaN,Collection: MP 1 bis,"P. Garelli RA 60 S. 124-125; Michel Innaya II,...",NaN,NaN,4395,NaN,https://aicuneiform.com/search?q=P361099,KIŠIB a-šùr-ma-lik DUMU i-na-a a-na a-šùr-na-d...,KIŠIB a-šùr-ma-lik DUMU i-na-a a-na a-šùr-na-d...
1,b05376c2-fc3d-49f8-9792-a25c0df9c383,https://oare.byu.edu/epigraphies/b05376c2-fc3d...,P359543,ICK 1 146,Cuneiform Tablet ICK 1 146,Ka 185 | Ka 455 | Ka 424?,"Cuneiform tablet, dated to the Old Assyrian Pe...",debt note,Ist Ka 185 | Ist Ka 455 | Ist Ka 424?,NaN,"ICK 1, 146 & ICK 2, 33 (env) + ICK 2, 163 (env)","Ichisar, Imdilum 65f.",NaN,NaN,1888,NaN,https://aicuneiform.com/search?q=P359543,0.33333 ma-na 7 GÍN KÙ.BABBAR ṣa-ru-pá-am i-ṣé...,0.33333 ma-na 7 GÍN KÙ.BABBAR ṣa-ru-pá-am i-ṣé...
2,80547963-f2a2-4d5d-9544-fc7ada60d3b2,https://oare.byu.edu/epigraphies/80547963-f2a2...,P361681,TrMA 1 | Or 36 396 n. 2,Cuneiform Tablet Or 36 396A.2,NaN,"Cuneiform tablet, dated to the Old Assyrian Pe...",debt note,TrMA 1,NaN,Or 36 396 n. 2,NaN,NaN,NaN,56,NaN,https://aicuneiform.com/search?q=P361681,{large break} a-na a-lá-lá-ma-sí ù a-mur-DINGI...,<big_gap> a-na a-lá-lá-ma-sí ù a-mur-DINGIR qí...
3,8c0d4238-f795-4061-8b3c-8469a66d86d6,https://oare.byu.edu/epigraphies/8c0d4238-f795...,P361098,MP 1,Cuneiform Tablet RA 60 123,RA 60 123,"Cuneiform tablet, dated to the Old Assyrian Pe...",letter,MP 1,NaN,NaN,NaN,NaN,NaN,4394,NaN,https://aicuneiform.com/search?q=P361098,um-ma PUZUR₄-a-šùr-ma a-na bu-za-zu qí-bi-ma a...,um-ma PUZUR₄-a-šùr-ma a-na bu-za-zu qí-bi-ma a...
4,2c0d870b-cd5e-fc36-8d81-114ecc7a9ac4,https://oare.byu.edu/epigraphies/2c0d870b-cd5e...,P290300,CCT 6 17a,Cuneiform Tablet CCT 6 17a (BM 115099),NaN,"Cuneiform tablet, dated to the Old Assyrian Pe...",unknown,BM 115099,NaN,"CCT 6, 17a","EL 320; 2-7: Garelli, AC 176A.4; 29-32: ebd. 3...",NaN,NaN,1603,NaN,https://aicuneiform.com/search?q=P290300,bu-za-zu il₅-we-da-ku iš-a-al-ma um-ma bu-za-z...,bu-za-zu il₅-we-da-ku iš-a-al-ma um-ma bu-za-z...


In [5]:
all_transliterations = published_texts_df['transliteration'].values
print(f'first line: {all_transliterations[0]}')
print(f'Number of lines: {len(all_transliterations)}')

first line: KIŠIB a-šùr-ma-lik DUMU i-na-a a-na a-šùr-na-da DUMU <big_gap> ù en-um-a-šùr DUMU <big_gap> a-pu-tum a-na a-wa-at ṭup-pì-im iḫ-da <big_gap> ša ú-<big_gap> ḫa-sí-sú <big_gap>
Number of lines: 7953


I will be using the `sentencepiece` framework to train the new tokenizer.

As practice, the training corpus must be organized as a one-sentence-per-line `.txt` file.

In [6]:
# with open(TOKENIZER_TRAINING_CORPUS_PATH, "w", encoding = 'utf-8') as f:
#     for line in all_transliterations:
#         f.write(line+"\n")

def get_tokenizer_training_corpus():
    return (
        all_transliterations[i : i + 1000]
        for i in range(0, len(all_transliterations), 1000)
    )

### Incremental training of old tokenizer

Incrementally train the old `mt5-base` tokenizer on new corpus.

In [46]:
old_tokenizer = AutoTokenizer.from_pretrained("models/mt5-base")

In [47]:
print(f'Old vocab size: {old_tokenizer.vocab_size}')

Old vocab size: 250100


Set the hyperparameters, mainly the new vocab size and the special tokens.

In [48]:
from collections import Counter

words = []
max_len = 0

for text_line in all_transliterations:
    max_len = max(max_len, len(text_line))
    words.extend(text_line.split())

word_counter = Counter(words)

print(f'There are {len(word_counter)} words (if split by space)')
print(word_counter.most_common(10))

print(f'The max sentence length (in bytes) = {max_len}')

There are 39868 words (if split by space)
[('a-na', 19844), ('ša', 18668), ('KÙ.BABBAR', 16167), ('<big_gap>', 15037), ('ma-na', 13929), ('DUMU', 9450), ('ù', 9400), ('GÍN', 9372), ('<gap>', 8561), ('i-na', 6937)]
The max sentence length (in bytes) = 4510


In [49]:
ACCADIAN_SPECIAL_TOKENS = [
    # placeholder for breaks
    "<gap>",          
    "<big_gap>",      
    # determinatives
    "{d}",            
    "{mul}",          
    "{ki}",           
    "{lu₂}",          
    "{e₂}",           
    "{uru}",          
    "{kur}",           
    "{mi}",            
    "{m}",             
    "{geš}",           
    "{tug₂}",          
    "{dub}",           
    "{id₂}",           
    "{mušen}",      
    "{na₄}",           
    "{kuš}",           
    "{u₂}",     
]

In [50]:
incremental_vocab_size = 16000

incremental_tokenizer = old_tokenizer.train_new_from_iterator(
    text_iterator = get_tokenizer_training_corpus(), 
    vocab_size = incremental_vocab_size, 
    new_special_tokens = ACCADIAN_SPECIAL_TOKENS 
)

In [51]:
incremental_tokenizer.vocab_size

16000

Sanity check

In [52]:
akkadian_text = "15 GÍN KÙ.BABBAR i-ṣé-er a-gu₅-tim i-dí-iš₈-tár i-šu iš-tù ḫa-mu-uš-tim ša a-bu-um-DINGIR a-na ša ni-ba-as i-ša-qal šu-ma lá iš-qú-ul 10 GÍN-tum 1 GÍN.TA i-na ITU i-ša-qal IGI ma-lá-ba IGI en-na-sú IGI tù-ra-am-ì-lí"

print(old_tokenizer.tokenize(akkadian_text))
print(incremental_tokenizer.tokenize(akkadian_text))

['▁15', '▁G', 'ÍN', '▁K', 'Ù', '.', 'BAB', 'BAR', '▁', 'i', '-', 'ṣ', 'é', '-', 'er', '▁', 'a', '-', 'gu', '5', '-', 'tim', '▁', 'i', '-', 'dí', '-', 'iš', '8', '-', 'tár', '▁', 'i', '-', 'šu', '▁iš', '-', 't', 'ù', '▁', 'ḫ', 'a', '-', 'mu', '-', 'uš', '-', 'tim', '▁', 'ša', '▁', 'a', '-', 'bu', '-', 'um', '-', 'DING', 'IR', '▁', 'a', '-', 'na', '▁', 'ša', '▁ni', '-', 'ba', '-', 'as', '▁', 'i', '-', 'ša', '-', 'qal', '▁', 'šu', '-', 'ma', '▁lá', '▁iš', '-', 'q', 'ú', '-', 'ul', '▁10', '▁G', 'ÍN', '-', 'tum', '▁1', '▁G', 'ÍN', '.', 'TA', '▁', 'i', '-', 'na', '▁', 'ITU', '▁', 'i', '-', 'ša', '-', 'qal', '▁', 'IGI', '▁ma', '-', 'lá', '-', 'ba', '▁', 'IGI', '▁en', '-', 'na', '-', 's', 'ú', '▁', 'IGI', '▁', 't', 'ù', '-', 'ra', '-', 'am', '-', 'ì', '-', 'lí']
['▁15', '▁GÍN', '▁KÙ.BABBAR', '▁i-ṣé-er', '▁a-gu5-t', 'im', '▁i-dí', '-iš8-tár', '▁i-šu', '▁iš-tù', '▁ḫa-mu-uš-t', 'im', '▁ša', '▁a-bu-um-DINGIR', '▁a-na', '▁ša', '▁ni', '-ba-as', '▁i-ša-qal', '▁šu-ma', '▁lá', '▁iš-qú-ul', '▁10', '▁GÍN

Note that the `incremental_tokenizer` itself does not include the learned tokens from `old_tokenizer`, so still need to manually update the old tokenier.

In [53]:
eng_text = "I love my jelly-bean with a spoon"

print(old_tokenizer.tokenize(eng_text))
print(incremental_tokenizer.tokenize(eng_text))

['▁I', '▁love', '▁my', '▁', 'jelly', '-', 'bean', '▁with', '▁', 'a', '▁', 'spoon']
['▁', 'I', '▁l', 'ov', 'e', '▁', 'm', 'y', '▁', 'j', 'e', 'l', 'l', 'y', '-be', 'an', '▁', 'wi', 't', 'h', '▁a', '▁s', 'p', 'oo', 'n']


In [54]:
# create a folder to store the finetuned model
os.makedirs(os.path.join(MODEL_FOLDER, MODEL_PREFIX), exist_ok=True)

# add the new tokens to the old tokenizer
new_tokens = list(set(incremental_tokenizer.vocab)-set(old_tokenizer.vocab))
old_tokenizer.add_tokens(new_tokens)

old_tokenizer.save_pretrained(os.path.join(MODEL_FOLDER, MODEL_PREFIX))

('models\\mt5-base-finetuned-english-akkadian\\tokenizer_config.json',
 'models\\mt5-base-finetuned-english-akkadian\\special_tokens_map.json',
 'models\\mt5-base-finetuned-english-akkadian\\spiece.model',
 'models\\mt5-base-finetuned-english-akkadian\\added_tokens.json',
 'models\\mt5-base-finetuned-english-akkadian\\tokenizer.json')

In [55]:
new_tokenizer = AutoTokenizer.from_pretrained(os.path.join(MODEL_FOLDER, MODEL_PREFIX))

new_tokenizer.vocab_size, len(new_tokenizer)

(250100, 265551)

In [56]:
print(old_tokenizer.tokenize(eng_text))
print(new_tokenizer.tokenize(eng_text))

['▁I', ' l', '▁', 'ove', '▁my', '▁', 'jelly', '-be', '▁an', '▁with', ' a', ' s', '▁po', 'on']
['▁I', ' l', '▁', 'ove', '▁my', '▁', 'jelly', '-be', '▁an', '▁with', ' a', ' s', '▁po', 'on']


In [45]:
print(akkadian_text)
print(old_tokenizer.tokenize(akkadian_text))
print(new_tokenizer.tokenize(akkadian_text))

15 GÍN KÙ.BABBAR i-ṣé-er a-gu₅-tim i-dí-iš₈-tár i-šu iš-tù ḫa-mu-uš-tim ša a-bu-um-DINGIR a-na ša ni-ba-as i-ša-qal šu-ma lá iš-qú-ul 10 GÍN-tum 1 GÍN.TA i-na ITU i-ša-qal IGI ma-lá-ba IGI en-na-sú IGI tù-ra-am-ì-lí
['▁15', ' GÍN', ' KÙ.BABBAR', ' i-ṣé-er', ' a-gu5-t', '▁im', ' i-dí-i', '▁', 'š', '8', '-tár', ' i-šu', ' iš-tù', ' ḫa-mu-uš-t', '▁im', ' ša', ' a-bu-um-DINGIR', ' a-na', ' ša', ' ni-ba', '-a', '▁', 's', ' i-ša-qal', ' šu-ma', ' l', '▁', 'á', ' iš-qú-ul', '▁10', ' GÍN', '-tum', '▁1', ' GÍN.TA', ' i-na', ' ITU', ' i-ša-qal', ' IGI', ' ma-lá', '-ba', ' IGI', ' en-na-sú', ' IGI', ' tù-ra-am-', 'ì-lí']
['▁15', ' GÍN', ' KÙ.BABBAR', ' i-ṣé-er', ' a-gu5-t', '▁im', ' i-dí-i', '▁', 'š', '8', '-tár', ' i-šu', ' iš-tù', ' ḫa-mu-uš-t', '▁im', ' ša', ' a-bu-um-DINGIR', ' a-na', ' ša', ' ni-ba', '-a', '▁', 's', ' i-ša-qal', ' šu-ma', ' l', '▁', 'á', ' iš-qú-ul', '▁10', ' GÍN', '-tum', '▁1', ' GÍN.TA', ' i-na', ' ITU', ' i-ša-qal', ' IGI', ' ma-lá', '-ba', ' IGI', ' en-na-sú', ' IGI', ' 

In [19]:
old_tokenizer.tokenize(eng_text)

['▁I',
 ' l',
 '▁',
 'ove',
 '▁my',
 '▁',
 'jelly',
 '-be',
 '▁an',
 '▁with',
 ' a',
 ' s',
 '▁po',
 'on']

In [20]:
akkadian_text

'15 GÍN KÙ.BABBAR i-ṣé-er a-gu₅-tim i-dí-iš₈-tár i-šu iš-tù ḫa-mu-uš-tim ša a-bu-um-DINGIR a-na ša ni-ba-as i-ša-qal šu-ma lá iš-qú-ul 10 GÍN-tum 1 GÍN.TA i-na ITU i-ša-qal IGI ma-lá-ba IGI en-na-sú IGI tù-ra-am-ì-lí'

In [21]:
old_tokenizer.tokenize(akkadian_text)

['▁15',
 ' GÍN',
 ' KÙ.BABBAR',
 ' i-ṣé-er',
 ' a-gu5-t',
 '▁im',
 ' i-dí-i',
 '▁',
 'š',
 '8',
 '-tár',
 ' i-šu',
 ' iš-tù',
 ' ḫa-mu-uš-t',
 '▁im',
 ' ša',
 ' a-bu-um-DINGIR',
 ' a-na',
 ' ša',
 ' ni-ba-',
 '▁as',
 ' i-ša-qal',
 ' šu-ma',
 ' l',
 '▁',
 'á',
 ' iš-qú-ul',
 '▁10',
 ' GÍN',
 '-tum',
 '▁1',
 ' GÍN.TA',
 ' i-na',
 ' ITU',
 ' i-ša-qal',
 ' IGI',
 ' ma-lá-',
 '▁ba',
 ' IGI',
 ' en-na-sú',
 ' IGI',
 ' tù-ra-am-',
 'ì-lí']

In [22]:
old_tokenizer.vocab_size, len(old_tokenizer)

(250100, 274770)

In [ ]:
# load the mt5 model and change the size of the embedding look-up table
mt5_model = AutoModel.from_pretrained(os.path.join(MODEL_FOLDER, "mt5-base"))
mt5_model.resize_token_embeddings(old_tokenizer.vocab_size)

In [198]:
mt5_model

MT5Model(
  (shared): Embedding(250112, 768)
  (encoder): MT5Stack(
    (embed_tokens): Embedding(250112, 768)
    (block): ModuleList(
      (0): MT5Block(
        (layer): ModuleList(
          (0): MT5LayerSelfAttention(
            (SelfAttention): MT5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): MT5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): MT5LayerFF(
            (DenseReluDense): MT5DenseGatedActDense(
              (wi_0): Linear(in_features=768, out_features=2048, bias=False)
              (wi_1): Linear(in_features=768, out_features=2048, bias=False)
              (wo): Linear(i

In [192]:
"<gap>" in set(incremental_tokenizer.vocab)-set(old_tokenizer.vocab)

True

In [138]:
old_tokenizer.tokenize(t2)

['▁I',
 ' l',
 '▁',
 'ove',
 '▁my',
 '▁',
 'jelly',
 '-be',
 '▁an',
 '▁with',
 ' a',
 ' s',
 '▁po',
 'on']

In [137]:
old_tokenizer.vocab_size

250100